# 03. Step-aware 강화학습 목적함수

## 학습 목표

- token별 policy ratio를 assistant step 단위 기하평균으로 바꿉니다.
- step -> trajectory 순서의 계층 평균이 긴 궤적 편향을 줄이는 원리를 확인합니다.
- 성공한 궤적의 key step에만 보너스를 주는 advantage shaping을 구현합니다.

실제 PPO/GRPO 훈련이 아니라 논문 수식의 동작을 숫자로 확인하는 toy simulation입니다.

In [ ]:
from math import exp, log
from statistics import mean, pstdev

def geometric_mean_ratio(token_ratios: list[float]) -> float:
    """Equation의 rho: 길이가 다른 step을 비교 가능한 하나의 비율로 만듭니다."""
    if not token_ratios or any(r <= 0 for r in token_ratios):
        raise ValueError("policy ratio는 양수여야 합니다")
    return exp(mean(log(r) for r in token_ratios))

def hierarchical_ratio(trajectories: list[list[list[float]]]) -> float:
    """각 step을 평균하고, 각 trajectory를 평균한 뒤 batch를 평균합니다."""
    trajectory_means = []
    for trajectory in trajectories:
        step_means = [geometric_mean_ratio(step) for step in trajectory]
        trajectory_means.append(mean(step_means))
    return mean(trajectory_means)

assert abs(geometric_mean_ratio([1.0, 1.0, 1.0]) - 1.0) < 1e-12

In [ ]:
# 짧은 궤적은 좋은 ratio, 긴 궤적은 약간 낮은 ratio를 갖도록 구성합니다.
short_trajectory = [[1.20, 1.10], [1.15, 1.05]]
long_trajectory = [[0.98, 0.99, 1.00, 0.97] for _ in range(12)]
trajectories = [short_trajectory, long_trajectory]

all_tokens = [
    ratio
    for trajectory in trajectories
    for step in trajectory
    for ratio in step
]
token_weighted = mean(all_tokens)
step_aware = hierarchical_ratio(trajectories)

print("token 수로 단순 평균:", round(token_weighted, 4))
print("step/trajectory 계층 평균:", round(step_aware, 4))
print("짧은 궤적도 batch에서 동일한 trajectory 비중을 가집니다.")

assert step_aware > token_weighted

In [ ]:
def group_relative_advantages(rewards: list[float]) -> list[float]:
    """같은 질문에서 sampling한 결과 보상을 z-score로 바꿉니다."""
    mu, sigma = mean(rewards), pstdev(rewards)
    if sigma == 0:
        return [0.0] * len(rewards)
    return [(reward - mu) / sigma for reward in rewards]

def shape_step_advantages(
    outcome_advantage: float,
    key_flags: list[bool],
    success: bool,
    lambda_key: float = 0.15,
) -> list[float]:
    """실패 궤적에는 key-step 보너스를 주지 않아 오류 행동 강화를 막습니다."""
    return [
        outcome_advantage + (lambda_key if success and is_key else 0.0)
        for is_key in key_flags
    ]

rewards = [1.0, 0.4, 0.0, 0.8]
advantages = group_relative_advantages(rewards)
key_flags = [False, True, False, True]

successful = shape_step_advantages(advantages[0], key_flags, success=True)
failed = shape_step_advantages(advantages[2], key_flags, success=False)
print("성공 궤적 step advantage:", [round(x, 3) for x in successful])
print("실패 궤적 step advantage:", [round(x, 3) for x in failed])

assert successful[1] > successful[0]
assert failed[1] == failed[0]

## 논문 ablation 수치 점검

전체 모델에서 각 구성 요소를 바꿨을 때의 절대 정확도 하락을 계산합니다. 이 값은 인과 효과의 완전한 분해가 아니라 논문의 통제된 단일 변경 실험 결과입니다.

In [ ]:
full_score = 82.5
ablations = {
    "직접 혼합 훈련": 77.5,
    "무작위 step replay": 74.1,
    "표준 GRPO": 79.4,
}

for name, score in ablations.items():
    print(f"{name:18s}: {full_score - score:4.1f}%p 하락")

step_losses = {
    "일반": 0.232,
    "증거 발견": 0.277,
    "경로 기각/전환": 0.298,
    "핵심 context update": 0.300,
}
ordinary = step_losses["일반"]
print("\n일반 step 대비 상대 손실 증가:")
for name, loss in step_losses.items():
    print(f"{name:20s}: {(loss / ordinary - 1):6.1%}")

assert round(full_score - ablations["무작위 step replay"], 1) == 8.4

## 확장 과제

1. PPO-style clipping을 추가하고 ratio가 clip 범위를 벗어날 때의 gradient 방향을 확인합니다.
2. assistant step 길이를 1-1000 token으로 무작위 생성해 token 평균과 계층 평균의 차이를 시각화합니다.
3. key-step label이 10% 잘못됐을 때 보너스가 결과를 얼마나 왜곡하는지 sensitivity test를 만듭니다.
4. 최종 정답뿐 아니라 출처 정확성, 제약 충족률, 검색 비용을 별도 reward로 측정합니다.